In [1]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from robustness_experiment_box.analysis.report_creator import ReportCreator
import re
import scipy.stats as st


In [2]:
df = pd.read_csv('/home/bosmanaw/Jasmin_explainabilitu/CFX-benchmarking/results/mnist_full_results_updated.csv')
print(df.columns)

metrics = ['IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility', 'morans_i_difference', 'optim_time']
data = "MNIST"

Index(['Unnamed: 0', 'network', 'image', 'original_label',
       'original_predicted_label', 'predicted_label', 'target', 'method',
       'IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility',
       'morans_i_original', 'morans_i_explanation',
       'morans_i_original_minus_explanation', 'optim_time',
       'correctness_ratio', 'weak_correctness', 'morans_i_difference',
       'timeout'],
      dtype='object')


In [14]:
df = pd.read_csv('/home/bosmanaw/Jasmin_explainabilitu/CFX-benchmarking/results/cifar_full_results.csv')
print(df.columns)

metrics = ['IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility', 'morans_i_difference', 'optim_time']
data = "CIFAR"

Index(['Unnamed: 0', 'network', 'image', 'original_label',
       'original_predicted_label', 'predicted_label', 'target', 'method',
       'IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility',
       'morans_i_original', 'morans_i_explanation',
       'morans_i_original_minus_explanation', 'optim_time',
       'correctness_ratio', 'weak_correctness', 'morans_i_difference',
       'timeout'],
      dtype='object')


In [63]:
aggregated = df.groupby('method').agg({
    'IM1': 'mean',
    'IM2': 'mean',
    'l2': 'mean',
    'implausibility': 'mean',
    'morans_i_difference':'mean',
   'optim_time': 'mean',
   'correctness_ratio':'mean'
})

aggregated['IM2'] = aggregated['IM2'].apply(lambda x: f"{x:.2e}")
aggregated = aggregated.round(2)
print(aggregated.to_latex(float_format="%.2f"))

\begin{tabular}{lrlrrrrr}
\toprule
 & IM1 & IM2 & l2 & implausibility & morans_i_difference & optim_time & correctness_ratio \\
method &  &  &  &  &  &  &  \\
\midrule
C-Min-Edit & 1.05 & 1.68e-03 & 16.16 & 1301.35 & 0.05 & 10.62 & 0.99 \\
Captum-MinParamPerturbation & 1.03 & 2.94e-03 & 6.54 & 1413.10 & 0.02 & 0.35 & 0.00 \\
Min-Edit & 1.04 & 1.66e-03 & 16.19 & 1302.51 & 0.05 & 6.83 & 0.98 \\
PIECE & 1.03 & 1.68e-03 & 16.43 & 1305.99 & 0.05 & 16.85 & 0.75 \\
alibi-CF & 1.08 & 8.20e-04 & 32.12 & 1787.46 & 0.01 & 107.99 & 0.92 \\
alibi-Proto-CF & 1.02 & 4.03e-03 & 36.70 & 1937.88 & 0.35 & 298.58 & 0.66 \\
\bottomrule
\end{tabular}



In [67]:
#the plot of all plots 
import itertools 

# Get the unique methods from the DataFrame
methods = df['method'].unique()
sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
# Loop over all pairs of methods (combinations)
for method1, method2 in itertools.combinations(methods, 2):
    for arch1, arch2 in itertools.combinations(df['network'].unique(), 2):
        # Filter data for the two methods
        df_method1 = df[(df['method'] == method1) & (df['network'].isin([arch1, arch2]))]
        df_method2 = df[(df['method'] == method2) & (df['network'].isin([arch1, arch2]))]


        # Merge the data on 'image' so we can compare both methods for the same image
        df_combined = pd.merge(df_method1[['image', 'network', 'correctness_ratio']], 
                            df_method2[['image', 'correctness_ratio']], 
                            on='image', 
                            suffixes=('_method1', '_method2'))
        df_counts = (
            df_combined.groupby(['correctness_ratio_method1', 'correctness_ratio_method2', 'network'])
            .size()
            .reset_index(name='count')
        )

        # Create the scatter plot
        plt.figure(figsize=(8, 6))  # Adjust the figure size if needed
        ax = sns.scatterplot(
            data=df_counts, 
            x='correctness_ratio_method1', 
            y='correctness_ratio_method2', 
            hue='network',  # Color points by network
            palette='tab10',  # Choose a color palette
            sizes=(20, 200),  # Adjust point size range
            size='count',
            alpha=0.7,  # Adjust transparency for overlapping points
        )

        max_value = max(df_combined['correctness_ratio_method1'].max(), 
                        df_combined['correctness_ratio_method2'].max())


        # Add plot details
        plt.xlabel(f'correctness ratio ({method1})', fontsize=12)
        plt.ylabel(f'correctness ratio ({method2})', fontsize=12)
        plt.legend(title="Network", fontsize=10, title_fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.xlim(-0.01, max_value +0.01)
        plt.ylim(-0.01, max_value+0.01)
        plt.plot([0, max_value], [0, max_value], color='red', linestyle='--', linewidth=2)
        sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


        
        # Save the plot (optional)
        plt.savefig(f'figures-xai/{data}/architecture/{data}_architecture{arch1}_{arch2}scatter_{method1}_vs_{method2}.pdf', dpi=300, bbox_inches='tight')
        plt.close()
    # Show the plot





In [ ]:

def boxplot_per_method(df):

    # Create a boxplot for each metric
    for metric in metrics:
        plt.figure(figsize=(12,12))
        sns.set(font_scale = 2)
        sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
        sns.set_palette('deep')

        
        # Create the boxplot
        sns.boxplot(
            data=df,
            x='method',
            y=metric,
            hue = 'method',
            legend = False
        )
        
        # Add titles and labels
        plt.xlabel('Method')
        plt.ylabel(metric)
        plt.xticks(rotation=45, ha='right')
    
        # Save the plot
        plt.savefig(f"figures-xai/{data}/boxplot/{data}_boxplot_{metric}.pdf", dpi=300, bbox_inches='tight')
        plt.close()
boxplot_per_method(df)

In [ ]:
import itertools
def scatter_plot_per_metric(df):
    plt.figure(figsize=(12,12))
    sns.set(font_scale = 2)
    sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
    # Select the relevant columns for the scatterplots
    hues = ['original_label', 'target', 'network', 'method']
    # Create scatterplots for each combination of metrics
    for x_metric, y_metric in itertools.combinations(metrics, 2):
        for hue in hues:

    
            scatter = sns.scatterplot(
                data=df, 
                x=x_metric, 
                y=y_metric, 
                hue=hue,
                palette='deep', 
                alpha=0.8
            )
            
            # Calculate and annotate Spearman correlation
            r, _ = spearmanr(df[x_metric], df[y_metric])
            plt.annotate(
                f"ρ={r:.2f}", 
                xy=(0.05, 0.95), 
                xycoords='axes fraction', 
                fontsize=12, 
                color='red', 
                ha='left', 
                va='top'
            )
            
            # Add plot labels and title
            plt.title(f'{y_metric} vs {x_metric}', fontsize=14)
            plt.xlabel(x_metric)
            plt.ylabel(y_metric)
            plt.legend(fontsize='small', title_fontsize='small')
            # Save the figure to a file
            filename = f"figures-xai/{data}/scatter/{data}_scatter_{y_metric}_vs_{x_metric}_in_{hue}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()


scatter_plot_per_metric(df)


In [15]:

import itertools
def scatterplot(df):
    plt.figure(figsize=(12,12))
    sns.set(font_scale = 2)
    sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
    sns.set_palette('deep')
    # Select the relevant columns for the scatterplots
    hues = ['original_label', 'target', 'network']
    metrics = ['IM1', 'implausibility']
    # Create scatterplots for each combination of metrics
    for x_method in itertools.combinations(df['method'].unique(), 1):
        x_metric = metrics[0]
        y_metric = metrics[1]
        for hue in hues:
 
    
            ax = scatter = sns.scatterplot(
                data=df[df.method == x_method[0]], 
                x=x_metric, 
                y=y_metric, 
                hue=hue,
                palette='deep', 
                alpha=0.8
            )
            
          
            # Add plot labels and title
            plt.title(f'{y_metric} vs {x_metric}', fontsize=14)
            plt.xlabel(x_metric)
            plt.ylabel(y_metric)
            plt.legend(fontsize='small', title_fontsize='small')
            ax.set(xscale="log", yscale="log")
            sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


            # Save the figure to a file
            filename = f"figures-xai/{data}/scatter/{data}_scatter_{x_method[0]}_in_{hue}.pdf"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()


scatterplot(df)

/scratch/bosmanaw/ipykernel_12400/1458097685.py:17: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.
  ax = scatter = sns.scatterplot(
/scratch/bosmanaw/ipykernel_12400/1458097685.py:31: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(fontsize='small', title_fontsize='small')
